# 과제 1 — 잘못된 SQL 고치기 (20점)

- 7 · 8주차 범위 · `과제1.ipynb` 를 9주차 수업 후 1주일 이내 LMS 에 제출
- 동료가 쓴 SQL 12개의 실수를 한 군데씩 고쳐 ✅ 를 받고, 무엇이 왜 틀렸는지 한 줄로 적습니다.
- 분석 질환은 모두 제2형 당뇨병(`'Diabetes Mellitus, Type 2'`)입니다.

| 채점 | 점수 | 내용 |
|---|---|---|
| 자동 확인 | 12점 | `check("Q1", _)` 가 ✅ 이면 1점 |
| 결과 해석 | 5점 | 숫자가 뜻하는 바를 문장으로 (해석1 3점 · 해석2 2점) |
| 실행 완료 · 형식 | 3점 | 모든 셀 실행 · 학번과 이름 · 문항마다 ‘무엇이 왜 틀렸나’ 한 줄 |

## 0. 시작하기 전에

- Colab 에서는 먼저 **`파일 → Drive에 사본 저장`** 을 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` 같은 줄은 지우지 마세요.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.
- ✏️ 칸은 첫 줄을 두고 `→` 뒤에 적습니다.

### 고치는 방법

1. 동료의 SQL 셀을 그대로 실행해 오류 메시지를 봅니다.
2. 실수 한 군데를 고쳐 다시 실행합니다.
3. 바로 아래 `check` 셀이 ✅ 이면 1점입니다.
4. ✏️ 칸에 한 줄로 적습니다. 예) `→ brief_titel 을 brief_title 로 고침 — 열 이름 철자가 틀림`

In [ ]:
#@title ✏️ 학번과 이름을 입력한 뒤 이 셀을 실행하세요 { display-mode: "form" }
학번 = ""  #@param {type:"string"}
이름 = ""  #@param {type:"string"}

In [ ]:
#@title ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

MESH = ["Diabetes Mellitus, Type 2"]
학번, 이름 = str(globals().get("학번", "")).strip(), str(globals().get("이름", "")).strip()
MY_MESH = MESH[int(_hl.md5(학번.encode()).hexdigest(), 16) % len(MESH)]      # 지금은 모든 학생 공통
if not 학번 or 학번 == "20260000":
    print("⚠️ 학번이 비어 있습니다. 맨 위 셀에 학번과 이름을 입력하고, 그 셀과 이 준비 셀을 다시 실행하세요.")
else:
    print(f"연결 완료.  {이름}({학번})")
print(f"분석 질환: {MY_MESH} (제2형 당뇨병, 모든 학생 공통)")

def 제출확인():
    """제출 전에 실행: 문항별 결과와 제출 방법을 보여 줍니다."""
    print(f"제출자: {이름 or '(이름 없음)'} ({학번 or '학번 없음'})  /  분석 질환: {MY_MESH}")
    진행확인()
    print("\n제출: ① 런타임 → 모두 실행 (오류에서 멈추면 아래 셀부터 '이후 셀 실행')  ② 파일 → 다운로드 → .ipynb  ③ LMS 에 과제1.ipynb 제출")

In [ ]:
#@title 자동 확인 도우미 — 정답 SQL 은 들어 있지 않습니다 (수정하지 마세요) { display-mode: "form" }
ANSWERS = _load_answers("eNqllu1uGkcYhW9lxB9aqYn2k92plEgUb5NIDiTEdhVFkbUfs60lYiog7Y8oEq5pRGyrhhQCTsDCqhvclLakIbUrUVXq5ewM95CZwW5mEUQi/gHsHBbxPnPO+84+iNyUI5+CB5H82pf0M2LqmuRC6DpQ0yOfgEh2hp7Lfpunukwv3WyGXd6J4JdHpFslh9XIXSp/tbZeYL9MW4n08rWla8krgLSLgDQGpD0ApF7C22XcHoKg/4Tsf88W24cXAfmzQ/Yr4L9jgHeK5NEO7p3QNd6qgeD4hEr4oAVw9eD0dl5hzkM55NG/8u1MHjHl68Jadt3OnEkPqXZTESkRdAzVQarpQU+knNSnUZLt1qjRJOVmiDLKyvuuA0a7vVH9KMpRL18CAhn9lnRbjCRKGv0L5HntAmkPo4Dt2tMqaQ0ZGulskK0389OpIp1i+I5i6JKHVFWkm9TnoKOWkc2NaVik/OupQbjWJ83BqEopGo+4x7WN+Uk0kUTyfEn1FCgZfiiNk/pUkkaPlF6xFG1uhGB4he0BflHC3R2A+7Xg38dnNNHPrevxRWsMyuLG43fe7Okik6PIpm5LjhLTNJFJVXTd0FQbeYYvMOki07pbWF3jyUTruWwmcw9RJJENd3/gTVZusY8F61ZC9OqLq1baAswr0i2SZ336DnC3ErzuAObhix7bENL8Pfh7unGF3P2ZjDGREUqubriaYtjQFBkn9TkSiJssZFHu3B/cilQ61F7VCh8lNHe0o9iKHPSDVwPSoFf7ZfJ0QOpDMKr/Mr99Rtg+w5cNqCPHlUU015BdTfNiih2TBDQoomW/QTk7k1nNF+zC/Ty7azrsLWvRSixR1CZ9DXG/OZ6aRXAlnVq+AT67TdcVvFsKjwsQT95eXYkvLp9a/CP3l9S3PmiimCK0bJuy6yAXIjfkp6ErihNDiu9BKPoZMjR481vQL86mxb2/eLc9OfwokVpOLn185hwVr8ZX+MHxzuccuke30QOXxrVy1HGumdMf1J5QRNWVmKNBW5J9VRJRJ/Vp0R3tPg6OO6R9Qp5XQojxFcpQL3NDG4OZ04e6m1wAZK8odOo5R48siXCaorua7NDjzfZFuEl9al/ulfBPO2MLQ3DjWumJHafFC41H7ZtWPIgmUtdvLFpL1kKUE/4/Zt+5PA+gHD7YTR8hzVdNiEIHBh26iiIbsuxps7qTnhOjzdbsnKaSvO1YxRSSniLMzvFEBvjn4UQzkm55tNcU9uKcIZVDDzAxn/ajYZqa4bkip61CRfV0VbKhPYsTbx8Fr0vv52RBpQ8lL/9hled5JB0g8IoPAJOg50jsw7fIXTY8")

## 데이터 구성 — 어떤 표가 있고, 어떻게 연결되나

열이 어느 표에 있는지, 표를 어떻게 잇는지 막히면 이 그림과 표를 보세요.

![aact_mini 표 구조 한눈에 보기](https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini_erd.png)

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | studies 와의 관계 | 주로 쓰는 열 |
|---|---|---|---|---|---|
| `studies` | 임상시험 1건 | `nct_id` | — | 기준이 되는 표 | brief_title, study_type, overall_status, phase, start_date, enrollment (숫자), enrollment_type (글자: ACTUAL · ESTIMATED), results_first_posted_date |
| `eligibilities` | 선정 · 제외 기준 | `id` | `nct_id` → studies | 1 : 1 (시험마다 1행) | gender, minimum_age (글자) |
| `sponsors` | 스폰서 1곳 | `id` | `nct_id` → studies | 1 : N (하나 이상) | agency_class, lead_or_collaborator, name |
| `conditions` | 대상 질환 1개 (연구자가 적은 이름) | `id` | `nct_id` → studies | 1 : N (하나 이상) | name (자유 입력이라 표기가 제각각) |
| `browse_conditions` | 표준 질환명(MeSH) 1개 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | mesh_term |
| `countries` | 참여 국가 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, removed (TRUE 면 등록 후 빠진 국가) |
| `facilities` | 수행 기관 1곳 | `id` | `nct_id` → studies | 1 : N (없을 수도 있음) | name, country |

**읽는 법** (5 · 6주차)

1. **기본키** — 한 행을 하나로 찾는 열입니다. `studies` 는 `nct_id`(임상시험 등록번호), 나머지 표는 `id` 입니다.
2. **외래키** — 다른 표의 `nct_id` 는 `studies.nct_id` 를 가리킵니다. 그래서 표를 연결(JOIN)할 때는 늘 `ON s.nct_id = 다른표.nct_id` 로 잇습니다.
3. **1 : N** — 시험 1건에 국가 · 기관 · 질환이 여러 개면 그 표에는 행이 여러 개입니다. 연결하면 시험 1건이 여러 행이 되므로, 시험 수는 `COUNT(DISTINCT s.nct_id)` 로 셉니다. 또 `browse_conditions` · `countries` · `facilities` 에는 행이 하나도 없는 시험도 있어서, INNER JOIN 하면 그 시험이 빠집니다.

In [ ]:
%%sql
-- 확인: studies 에서는 1행인 시험이 countries 와 연결하면 국가 수만큼 여러 행이 됩니다
SELECT s.nct_id, s.overall_status, c.name AS 국가
FROM studies AS s
JOIN countries AS c ON s.nct_id = c.nct_id
WHERE s.nct_id = 'NCT01795521'

## 1. 조회 기초 (7주차) — Q1~Q6

**Q1.** 동료가 원한 것: 지금 모집 중(`RECRUITING`)인 시험이 몇 건인지 세어 주세요.

💡 힌트: 문자 값은 어떻게 감싸나요? (7주차 자주 하는 실수)

In [ ]:
%%sql
-- Q1
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 모집중
FROM studies
WHERE overall_status = RECRUITING;

In [ ]:
check("Q1", _)

✏️ **Q1 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q2.** 동료가 원한 것: 2023년 1월 1일부터(그날 포함) 시작한 시험이 몇 건인지 세어 주세요.

💡 힌트: 날짜 값은 어떤 모양으로 적고, 무엇으로 감싸나요? (7주차 WHERE 조건에 쓰는 기호)

In [ ]:
%%sql
-- Q2
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE start_date >= 2023;

In [ ]:
check("Q2", _)

✏️ **Q2 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q3.** 동료가 원한 것: 등록 인원이 1000명 이상인 시험이 몇 건인지 세어 주세요.

💡 힌트: 숫자 열과 비교하는 값은 어떻게 적나요? (7주차 WHERE 조건에 쓰는 기호)

In [ ]:
%%sql
-- Q3
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE enrollment >= '1000명';

In [ ]:
check("Q3", _)

✏️ **Q3 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q4.** 동료가 원한 것: 여성만 참여할 수 있는 시험(`gender` 가 `FEMALE`)이 몇 건인지 세어 주세요.

💡 힌트: gender 열은 어느 표에 있나요? (위 데이터 구성 · 7주차 ERD)

In [ ]:
%%sql
-- Q4
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 여성대상
FROM studies
WHERE gender = 'FEMALE';

In [ ]:
check("Q4", _)

✏️ **Q4 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q5.** 동료가 원한 것: 완료(`COMPLETED`)된 시험 중 등록 인원이 가장 많은 5건을 많은 순서대로 보여 주세요.

💡 힌트: 절을 적는 순서는 어떻게 되나요? (7주차 자주 하는 실수)

In [ ]:
%%sql
-- Q5
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT nct_id, enrollment
FROM studies
ORDER BY enrollment DESC
WHERE overall_status = 'COMPLETED'
LIMIT 5;

In [ ]:
check("Q5", _)

✏️ **Q5 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q6.** 동료가 원한 것: 3상 또는 4상(`PHASE3`, `PHASE4`) 시험이 몇 건인지 세어 주세요.

💡 힌트: OR 로 잇는 두 조건이 각각 완전한가요? (7주차 자주 하는 실수)

In [ ]:
%%sql
-- Q6
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(*) AS 시험수
FROM studies
WHERE phase = 'PHASE3' OR 'PHASE4';

In [ ]:
check("Q6", _)

✏️ **Q6 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 2. 집계와 GROUP BY (8주차) — Q7~Q9

**Q7.** 동료가 원한 것: 진행 상태(`overall_status`)별로 시험이 몇 건인지 세어 주세요.

💡 힌트: COUNT 와 일반 열을 같이 쓰면 무엇이 더 필요한가요? (8주차 자주 하는 실수)

In [ ]:
%%sql
-- Q7
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT overall_status, COUNT(*) AS 시험수
FROM studies;

In [ ]:
check("Q7", _)

✏️ **Q7 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q8.** 동료가 원한 것: 제외된 국가 행은 빼고, 참여 시험이 1000건 이상인 국가만 시험 수를 많은 순으로 보여 주세요.

💡 힌트: 집계 결과에 거는 조건은 어느 절에 쓰나요? (8주차 자주 하는 실수)

In [ ]:
%%sql
-- Q8
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT name AS 국가, COUNT(DISTINCT nct_id) AS 시험수
FROM countries
WHERE removed = false
  AND COUNT(DISTINCT nct_id) >= 1000
GROUP BY name
ORDER BY 시험수 DESC;

In [ ]:
check("Q8", _)

✏️ **Q8 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q9.** 동료가 원한 것: 관찰 연구(`OBSERVATIONAL`)의 평균 등록 인원(명)을 소수 첫째 자리까지 보여 주세요.

💡 힌트: AVG 는 어떤 타입의 열에 쓸 수 있나요? (8주차 평균과 중앙값 · 위 데이터 구성)

In [ ]:
%%sql
-- Q9
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT ROUND(AVG(enrollment_type), 1) AS 평균인원
FROM studies
WHERE study_type = 'OBSERVATIONAL';

In [ ]:
check("Q9", _)

✏️ **Q9 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 3. 당뇨병 시험 분석 (JOIN) — Q10~Q12

**Q10.** 동료가 원한 것: 제2형 당뇨병 시험 중 완료(`COMPLETED`)된 시험이 몇 건인지 세어 주세요.

💡 힌트: 두 표에 모두 있는 열은 어느 표의 것인지 어떻게 밝히나요? (8주차 JOIN)

In [ ]:
%%sql
-- Q10
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT COUNT(DISTINCT nct_id) AS 완료시험
FROM studies AS s
JOIN browse_conditions AS b ON s.nct_id = b.nct_id
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2'
  AND s.overall_status = 'COMPLETED';

In [ ]:
check("Q10", _)

✏️ **Q10 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q11.** 동료가 원한 것: 제2형 당뇨병 시험을 진행 상태별로 나누어 시험 수를 많은 순으로 보여 주세요.

💡 힌트: 두 표를 어떤 열로 맞출지는 어디에 적나요? (8주차 JOIN — 두 표 연결하기)

In [ ]:
%%sql
-- Q11
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT s.overall_status AS 상태, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2'
GROUP BY s.overall_status
ORDER BY 시험수 DESC;

In [ ]:
check("Q11", _)

✏️ **Q11 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

**Q12.** 동료가 원한 것: 제2형 당뇨병 시험을 임상 단계(`phase`)별로 나누어 시험 수를 많은 순으로 보여 주세요.

💡 힌트: 표에 별칭(AS s)을 붙이면, 그 뒤로는 표를 무엇으로 부르나요? (8주차 용어 정리 · 별칭)

In [ ]:
%%sql
-- Q12
-- [고칠 SQL] 실수 한 군데를 고쳐 실행하세요
SELECT s.phase AS 단계, COUNT(DISTINCT s.nct_id) AS 시험수
FROM studies AS s
JOIN browse_conditions AS b ON studies.nct_id = b.nct_id
WHERE b.mesh_term = 'Diabetes Mellitus, Type 2'
GROUP BY s.phase
ORDER BY 시험수 DESC;

In [ ]:
check("Q12", _)

✏️ **Q12 무엇이 왜 틀렸나** (두 번 눌러 한 줄로):
→ 

## 4. 결과 해석 (5점)

위 Q10~Q12 의 내 결과 숫자로 문장을 쓰세요.

**해석1 (3점) — 제2형 당뇨병 임상시험 현황**

1. 당뇨병 시험은 모두 몇 건인가요? (Q11 시험수의 합)
2. 그중 완료된 시험(Q10)은 몇 건, 몇 % 인가요? (Q10 ÷ 1번 값 × 100, 소수 첫째 자리)
3. Q11 에서 모집 중(`RECRUITING`)인 시험은 몇 건인가요? 환자 수인가요, 시험 수인가요?

✏️ **해석1** (두 번 눌러 2~3문장으로):
→ 

**해석2 (2점) — 검산**

1. Q12 의 시험수를 모두 더하면 몇 건인가요? (`NA` · `None` 행도 더합니다)
2. 이 합계는 해석1 의 1번 값과 같나요? 같아야 하는 이유는? (9주차 검산)

✏️ **해석2** (두 번 눌러 한두 문장으로):
→ 

## 5. 제출 전 확인

- [ ] 맨 위 셀에 학번과 이름을 입력했다
- [ ] Q1~Q12 가 모두 ✅ 이다
- [ ] ✏️ 칸(Q1~Q12 · 해석1 · 해석2)을 모두 적었다
- [ ] 모든 셀을 실행한 상태로 저장했다

In [ ]:
제출확인()